In [1]:
import json
import re
import urllib.request
import time
from pathlib import Path
from IPython.display import display, Markdown

# =========================================================
# CONFIG
# =========================================================
BASE_DIR     = Path(".")
TEXTS_DIR    = BASE_DIR / "data_texts"
FR_PATH      = TEXTS_DIR / "20000lieues_fr.txt"
JSON_PATH    = BASE_DIR / "Points-escales-chapitres.json"
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "mistral"

# ── TEST : 3 escales avec chapitres différents ──
TEST_ESCALES = ["Île de Crespo", "Archipel des Pomotou", "Méditerranée"]

fr_text_full = FR_PATH.read_text(encoding="utf-8", errors="ignore")
fr_text_full = fr_text_full.replace("\r\n", "\n").replace("\r", "\n")

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"📖 FR : {len(fr_text_full):,} caractères")
print(f"🗺️  {len(data)} escales dans le JSON")

# =========================================================
# EXTRACTION DU CHAPITRE FR
# =========================================================
ROMAINS_FR = {
    1: "PREMIER", 2: "II", 3: "III", 4: "IV", 5: "V",
    6: "VI", 7: "VII", 8: "VIII", 9: "IX", 10: "X",
    11: "XI", 12: "XII", 13: "XIII", 14: "XIV", 15: "XV",
    16: "XVI", 17: "XVII", 18: "XVIII", 19: "XIX", 20: "XX",
    21: "XXI", 22: "XXII", 23: "XXIII", 24: "XXIV"
}

def extraire_chapitre_fr(texte, partie, numero):
    romain = ROMAINS_FR.get(numero, str(numero))
    marqueur = f"CHAPITRE {romain}"
    split_partie = texte.split("FIN DE LA PREMIÈRE PARTIE")
    zone = split_partie[0] if partie == 1 else (split_partie[1] if len(split_partie) > 1 else texte)
    idx_debut = zone.find(marqueur)
    if idx_debut == -1: return None
    idx_suite = zone.find("CHAPITRE", idx_debut + len(marqueur))
    return zone[idx_debut:idx_suite].strip() if idx_suite != -1 else zone[idx_debut:].strip()

print("✅ Utilitaires chargés")

# =========================================================
# OLLAMA
# =========================================================
def appeler_ollama(prompt, num_predict=1500):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {"temperature": 0.1, "num_predict": num_predict}
    }).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL, data=payload,
        headers={"Content-Type": "application/json"}, method="POST"
    )
    try:
        with urllib.request.urlopen(req, timeout=180) as resp:
            result = json.loads(resp.read().decode("utf-8"))
            return result.get("response", "").strip()
    except Exception as e:
        print(f"  ❌ Ollama erreur : {e}")
        return None

def prompt_especes(nom_escale, chapitre_texte):
    return f"""Tu es un naturaliste du XIXe siècle spécialisé en biologie marine et expert de Jules Verne.

Voici un chapitre de "Vingt mille lieues sous les mers" correspondant à l'escale : {nom_escale}

TEXTE :
{chapitre_texte[:6000]}

INSTRUCTIONS STRICTES :
- Relève UNIQUEMENT les espèces qui vivent DANS L'EAU MARINE : poissons, mollusques, crustacés, céphalopodes, échinodermes, algues, coraux, méduses, cétacés, poulpes, pieuvres, etc.
- EXCLUS absolument : oiseaux, tortues terrestres, mammifères terrestres, plantes terrestres, insectes — même s'ils sont mentionnés dans le texte.
- Relève uniquement ce qui est explicitement cité dans le texte.
- Ne cite que ce qui est écrit noir sur blanc — zéro invention, zéro ajout.
- La citation doit être le passage EXACT où l'espèce est nommée — pas une phrase générique du texte.
- Si tu ne trouves pas de citation directe pour une espèce, ne l'inclus pas.
- Si aucune espèce marine n'est citée, réponds UNIQUEMENT : "Aucune espèce marine identifiée dans ce passage. Jules Verne décrit ici plutôt la géographie ou les événements du voyage."
- Réponds UNIQUEMENT en français.
- Pour CHAQUE espèce sans exception, donne exactement ce format sur 3 lignes :

• [Nom exact cité par Jules Verne] ([Nom latin scientifique — cherche dans tes connaissances biologiques, omet les parenthèses si vraiment inconnu])
  Catégorie : Faune marine OU Flore marine
  Jules Verne écrit : "[citation exacte de moins de 15 mots tirée du texte]"

- Ne liste JAMAIS une espèce sans sa citation et sa catégorie — format complet obligatoire.

- Groupe les espèces en deux sections :
  ━━ FAUNE MARINE ━━
  ━━ FLORE MARINE ━━
- Si une seule catégorie est présente, n'affiche que celle-là.
"""

# =========================================================
# TEST SUR 3 ESCALES
# =========================================================
for escale in data:
    nom_fr = escale.get("Escales du Nautilus", {}).get("fr", "")
    
    if nom_fr not in TEST_ESCALES:
        continue
    
    ch_info = escale.get("chapitre")
    if not ch_info:
        continue
    
    partie = ch_info["partie"]
    numero = ch_info["chapitre"]
    
    print(f"\n{'='*50}")
    print(f"🧭 {nom_fr} — Partie {partie}, Chapitre {numero}")
    print(f"{'='*50}")
    
    ch_fr = extraire_chapitre_fr(fr_text_full, partie, numero)
    if not ch_fr:
        print("❌ Chapitre introuvable")
        continue
    
    print(f"📄 Chapitre extrait : {len(ch_fr)} caractères")
    print(f"🤖 Envoi à Ollama...")
    
    reponse = appeler_ollama(prompt_especes(nom_fr, ch_fr))
    
    display(Markdown(f"""
---
### 🧭 {nom_fr}
{reponse}
---
"""))
    time.sleep(2)

print("\n✅ Test terminé !")


📖 FR : 908,396 caractères
🗺️  32 escales dans le JSON
✅ Utilitaires chargés

🧭 Île de Crespo — Partie 1, Chapitre 15
📄 Chapitre extrait : 16671 caractères
🤖 Envoi à Ollama...



---
### 🧭 Île de Crespo
━━ FAUNE MARINE ━━

• Jambonneaux (spécimen non identifié)
  Catégorie : Crustacés
  Jules Verne écrit : "Autrefois, on en faisait de belles étoffes, des bas, des gants, car ils étaient à la fois très-moelleux et très-chauds."

• Cladostèphes verticillées (spécimen non identifié)
  Catégorie : Algues
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je remarquai des cladostèphes verticillées."

• Padines-paon (spécimen non identifié)
  Catégorie : Algues
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je remarquai des padines-paon."

• Caulerpes à feuilles de vigne (spécimen non identifié)
  Catégorie : Algues
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je remarquai des caulerpes à feuilles de vigne."

• Callithamnes granifères (spécimen non identifié)
  Catégorie : Algues
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je remarquai des callithamnes granifères."

• Céramies à teintes écarlates (spécimen non identifié)
  Catégorie : Algues
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je remarquai des délicates céramies à teintes écarlates."

• Agares (spécimen non identifié)
  Catégorie : Algues
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je remarquai des agares disposées en éventails."

• Acétabules (spécimen non identifié)
  Catégorie : Fungi marins
  Jules Verne écrit : "Parmi ces précieuses hydrophytes, je remarquai des acétabules, semblables à des chapeaux de champignons très-déprimés."

• Cétacés (spécimen non identifié)
  Catégorie : Mammifères marins
  Jules Verne écrit : "La direction du _Nautilus_ se maintint à l'est-nord-est, sa vitesse à douze milles, sa profondeur entre cinquante et soixante mètres." (Cet extrait ne mentionne pas explicitement les cétacés mais il est possible qu'ils soient présents dans la faune marine de l'environnement du Nautilus)

• Poulpes (spécimen non identifié)
  Catégorie : Céphalopodes
  Jules Verne écrit : "Mais que faisait le vent à ce _Nautilus_ que les tempêtes ne pouvaient effrayer!" (Cet extrait ne mentionne pas explicitement les poulpes mais il est possible qu'ils soient présents dans la faune marine de l'environnement du Nautilus)

• Pieuvres (spécimen non identifié)
  Catégorie : Céphalopodes
  Jules Verne écrit : "Mais que faisait le vent à ce _Nautilus_ que les tempêtes ne pouvaient effrayer!" (Cet extrait ne mentionne pas explicitement les pieuvres mais il est possible qu'elles soient présentes dans la faune marine de l'environnement du Nautilus)

━━ FLORE MARINE ━━

• Zostère marine (Zostera marina)
  Catégorie : Algues
  Jules Verne écrit : "Le 11 novembre, je commençai le journal de ces aventures, ce qui m'a permis de les raconter avec la plus scrupuleuse exactitude, et, détail curieux, je l'écrivis sur un papier fabriqué avec la zostère marine."
---



🧭 Archipel des Pomotou — Partie 1, Chapitre 19
📄 Chapitre extrait : 21056 caractères
🤖 Envoi à Ollama...



---
### 🧭 Archipel des Pomotou
━━ FAUNE MARINE ━━

• Poissons (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "nous apercevions souvent des coques naufragées qui achevaient de pourrir entre deux eaux, et, plus profondément, des canons, des boulets, des ancres, des chaînes, et mille autres objets de fer, que la rouille dévorait."

• Cétacés (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "Les hasards de sa navigation avaient précisément conduit le _Nautilus_ vers l'île Clermont-Tonnerre, l'une des plus curieuses du groupe."

• Poulpes (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "Je pus observer de très-près ces curieuses murailles, car, à leur aplomb, la sonde accusait plus de trois cents mètres de profondeur."

• Pieuvres (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "Je pus observer de très-près ces curieuses murailles, car, à leur aplomb, la sonde accusait plus de trois cents mètres de profondeur."

• Crustacés (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "Ces îles sont coralligènes. Un soulèvement lent, mais continu, provoqué par le travail des polypes, les reliera un jour entre elles."

• Céphalopodes (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "Je pus observer de très-près ces curieuses murailles, car, à leur aplomb, la sonde accusait plus de trois cents mètres de profondeur."

• Échinodermes (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "Ces murailles étaient spécialement l'œuvre des madréporaires désignés par les noms de millepores, de porites, d'astrées et de méandrines."

• Méduses (Nom latin inconnu)
  Catégorie : Faune marine
  Jules Verne écrit : "Je pus observer de très-près ces curieuses murailles, car, à leur aplomb, la sonde accusait plus de trois cents mètres de profondeur."

Aucune espèce marine identifiée dans ce passage. Jules Verne décrit ici plutôt la géographie ou les événements du voyage.
---



🧭 Méditerranée — Partie 2, Chapitre 7
📄 Chapitre extrait : 22108 caractères
🤖 Envoi à Ollama...



---
### 🧭 Méditerranée
━━ FAUNE MARINE ━━

• Lamproie (Espèce inconnue)
  Catégorie : Poisson osseux
  Jules Verne écrit : "des lamproies longues d'un mètre"

• Oxyrhynque (Raja clavata)
  Catégorie : Raie
  Jules Verne écrit : "des oxyrhinques, sortes de raies"

• Squale-milandre (Squalus albipinnis)
  Catégorie : Requin cartilagineux
  Jules Verne écrit : "des squales-milandres, longs de douze pieds"

• Renard marin (Chimaera monstrosa)
  Catégorie : Chimère
  Jules Verne écrit : "des renards marins, longs de huit pieds"

• Dorade (Sparus aurata)
  Catégorie : Poisson osseux
  Jules Verne écrit : "des dorades, du genre spare"

• Esturgeon (Scaphirhynchus platorynchus)
  Catégorie : Esturgeon cartilagineux
  Jules Verne écrit : "des esturgeons magnifiques, longs de neuf à dix mètres"

• Scombre-thon (Thunnus thynnus)
  Catégorie : Thon
  Jules Verne écrit : "des scombres-thons, au dos bleu-noir"
---



✅ Test terminé !
